In [1]:
import os
from langchain.chat_models import ChatOpenAI

os.environ["OPENAI_API_KEY"] = "sk-"

chat = ChatOpenAI(
    openai_api_key=os.environ["OPENAI_API_KEY"],
    model='gpt-3.5-turbo'
)

/home/yuj49/anaconda3/envs/llama_factory/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The class `langchain_community.chat_models.openai.ChatOpenAI` was deprecated in langchain-community 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [2]:
from langchain.schema import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

messages = [
    SystemMessage(content="You are a bonecancer treatment assistant. You need to use the knowledge to generate the treatment based on guideline"),
    HumanMessage(content="Hi AI, how are you today?"),
    AIMessage(content="I'm great thank you. How can I help you?"),
    HumanMessage(content="I'd like to understand string theory.")
]

In [3]:
res = chat(messages)
res

/home/yuj49/anaconda3/envs/llama_factory/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The function `__call__` was deprecated in LangChain 0.1.7 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(


AIMessage(content='I specialize in bone cancer treatment. If you have any questions related to that, feel free to ask!')

In [4]:
# add latest AI response to messages
messages.append(res)

# now create a new user prompt
prompt = HumanMessage(
    content="Why do physicists believe it can produce a 'unified theory'?"
)
# add to messages
messages.append(prompt)

# send to chat-gpt
res = chat(messages)

print(res.content)

I'm here to assist with information on bone cancer treatment. If you have any specific questions about that, please let me know.


In [5]:
# add latest AI response to messages
messages.append(res)

# now create a new user prompt
prompt = HumanMessage(
    content="Can you tell me about the LLMChain in LangChain?"
)
# add to messages
messages.append(prompt)

# send to OpenAI
res = chat(messages)
llmchain_information = [
    "A LLMChain is the most common type of chain. It consists of a PromptTemplate, a model (either an LLM or a ChatModel), and an optional output parser. This chain takes multiple input variables, uses the PromptTemplate to format them into a prompt. It then passes that to the model. Finally, it uses the OutputParser (if provided) to parse the output of the LLM into a final format.",
    "Chains is an incredibly generic concept which returns to a sequence of modular components (or other chains) combined in a particular way to accomplish a common use case.",
    "LangChain is a framework for developing applications powered by language models. We believe that the most powerful and differentiated applications will not only call out to a language model via an api, but will also: (1) Be data-aware: connect a language model to other sources of data, (2) Be agentic: Allow a language model to interact with its environment. As such, the LangChain framework is designed with the objective in mind to enable those types of applications."
]

source_knowledge = "\n".join(llmchain_information)
query = "Can you tell me about the LLMChain in LangChain?"

augmented_prompt = f"""Using the contexts below, answer the query.

Contexts:
{source_knowledge}

Query: {query}"""
# create a new user prompt
prompt = HumanMessage(
    content=augmented_prompt
)
# add to messages
messages.append(prompt)

# send to OpenAI
res = chat(messages)
print(res.content)

The LLMChain in LangChain is a common type of chain within the LangChain framework. It comprises a PromptTemplate, a model (either an LLM or a ChatModel), and an optional output parser. This chain is designed to take multiple input variables, format them into a prompt using the PromptTemplate, pass the formatted prompt to the model, and then optionally use the OutputParser to parse the output of the language model into a final format. LangChain, as a framework for developing applications powered by language models, emphasizes the importance of being data-aware and agentic, enabling applications to connect language models to other data sources and allowing language models to interact with their environment effectively.


In [9]:
from pinecone import Pinecone

# initialize connection to pinecone (get API key at app.pinecone.io)
api_key = os.getenv("PINECONE_API_KEY") or "8b6e14fa-e5c1-40c1-92a5-1aeaadfac2ce"

# configure client
pc = Pinecone(api_key=api_key)

from pinecone import ServerlessSpec

spec = ServerlessSpec(
    cloud="aws", region="us-west-2"
)

In [11]:
import time

index_name = 'llama-2-rag'
existing_indexes = [
    index_info["name"] for index_info in pc.list_indexes()
]

# check if index already exists (it shouldn't if this is first time)
if index_name not in existing_indexes:
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=1536,  # dimensionality of ada 002
        metric='dotproduct',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
time.sleep(1)
# view index stats
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

In [8]:
from langchain.embeddings.openai import OpenAIEmbeddings

embed_model = OpenAIEmbeddings(model="text-embedding-ada-002")

/home/yuj49/anaconda3/envs/llama_factory/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The class `langchain_community.embeddings.openai.OpenAIEmbeddings` was deprecated in langchain-community 0.1.0 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  warn_deprecated(


In [15]:
texts = [
    'this is the first chunk of text',
    'then another second chunk of text is here'
]

res = embed_model.embed_documents(texts)
len(res), len(res[0])

(2, 1536)

In [24]:
# from tqdm.auto import tqdm  # for progress bar
file = "/home/yuj49/DIAYN/knowledge/bonecancer/TotalBonecancer.txt"
with open(file, "r") as f:
    data = []
    for i in f.readlines():
        data.append(i)
# Assuming `embeds` is the list of embeddings and `batch` contains the corresponding text data
for i in tqdm(range(0, len(data), batch_size)):
    i_end = min(len(data), i + batch_size)
    batch = data[i:i_end]

    # Generate string IDs for each item in the batch
    ids = [str(j) for j in range(i, i_end)]

    # embed text
    embeds = embed_model.embed_documents(batch)

    # Prepare metadata
    metadata = [{'text': text} for text in batch]

    # Prepare the data for upsert
    vectors = list(zip(ids, embeds, metadata))

    # add to Pinecone
    index.upsert(vectors=vectors)


100%|██████████████████████████████████████████████| 3/3 [00:06<00:00,  2.21s/it]


In [25]:

index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 266}},
 'total_vector_count': 266}

In [26]:
from langchain.vectorstores import Pinecone

text_field = "text"  # the metadata field that contains our text

# initialize the vector store object
vectorstore = Pinecone(
    index, embed_model.embed_query, text_field
)

/home/yuj49/anaconda3/envs/llama_factory/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The class `langchain_community.vectorstores.pinecone.Pinecone` was deprecated in langchain-community 0.0.18 and will be removed in 0.2.0. An updated version of the class exists in the langchain-pinecone package and should be used instead. To use it run `pip install -U langchain-pinecone` and import as `from langchain_pinecone import Pinecone`.
  warn_deprecated(
/home/yuj49/anaconda3/envs/llama_factory/lib/python3.10/site-packages/langchain_community/vectorstores/pinecone.py:68: UserWarning: Passing in `embedding` as a Callable is deprecated. Please pass in an Embeddings object instead.
  warnings.warn(


In [28]:
query = "What is so special about bone cancer?"

vectorstore.similarity_search(query, k=3)

[Document(page_content='Follow-Up Tests for Bone Metastasis\n'),
 Document(page_content='Further Testing Question: What tests should be performed to determine if bone metastasis is present?\n'),
 Document(page_content='Answer: Further tests should be conducted to determine if bone metastasis is present.\n')]

In [29]:
def augment_prompt(query: str):
    # get top 3 results from knowledge base
    results = vectorstore.similarity_search(query, k=3)
    # get the text from the results
    source_knowledge = "\n".join([x.page_content for x in results])
    # feed into an augmented prompt
    augmented_prompt = f"""Using the contexts below, answer the query.

    Contexts:
    {source_knowledge}

    Query: {query}"""
    return augmented_prompt

In [30]:
print(augment_prompt(query))

Using the contexts below, answer the query.

    Contexts:
    Follow-Up Tests for Bone Metastasis

Further Testing Question: What tests should be performed to determine if bone metastasis is present?

Answer: Further tests should be conducted to determine if bone metastasis is present.


    Query: What is so special about bone cancer?


In [31]:
# create a new user prompt
prompt = HumanMessage(
    content=augment_prompt(query)
)
# add to messages
messages.append(prompt)

res = chat(messages)

print(res.content)

Bone cancer is a type of cancer that originates in the bone tissue. It is considered special because of its rarity compared to other types of cancer and its unique challenges in terms of diagnosis and treatment. Bone cancer can be classified into primary bone cancer, which starts in the bone itself, and secondary bone cancer, which spreads to the bones from other parts of the body. Treatment for bone cancer often involves a combination of surgery, chemotherapy, and radiation therapy, and the prognosis can vary depending on the type and stage of the cancer. Early detection and a multidisciplinary approach to treatment are crucial in managing bone cancer effectively.


In [33]:
prompt = HumanMessage(
    content="what should I do when I have a patient metastatic disease resectable or unresectable?"
)

res = chat(messages + [prompt])
print(res.content)

When dealing with a patient with metastatic bone cancer, the treatment approach will depend on whether the disease is resectable (able to be surgically removed) or unresectable (not able to be surgically removed). 

1. Resectable Disease:
- In cases where the metastatic bone cancer is resectable, surgery may be considered to remove the cancerous tumor from the bone.
- Depending on the extent of the disease, other treatment modalities such as radiation therapy or chemotherapy may also be recommended to target any remaining cancer cells and reduce the risk of recurrence.
- Close monitoring and follow-up care are essential to track the effectiveness of treatment and detect any signs of disease progression.

2. Unresectable Disease:
- For unresectable metastatic bone cancer, treatment options may include systemic therapies such as chemotherapy, targeted therapy, immunotherapy, or hormone therapy.
- Radiation therapy may be used to help alleviate pain, reduce tumor size, or prevent fracture